Grouping
---
In this lesson we will go over the split-apply-combine strategy and the groupby() function.

## Learning objectives
By the end ofthis lesson, we will be able to:

* Understand and apply the Split-Apply-Combine strategy to analyze grouped data.
* Use groupby() to split a pandas.DataFrame by one or more columns.
* Calculate summary statistics for groups in a pandas.DataFrame.
* Use method chaining for readable data analysis.
* Explain how groupby() handles NA values and choose between count() and size()   to count observations in each group.

In [14]:
# Import python libraries and load the penguin data
import numpy as np
import pandas as pd

# Load Palmer penguins data
URL = 'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv'
penguins = pd.read_csv(URL)

penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


## Summary statistics
It is easy to get summary statistics for each column in a `pandas.DataFrame` by using methods such as:

* sum(): sum values in each column,
* count(): count non-NA values in each column,
* size(): count the number of values in a column (including NA values),
* min() and max(): get the minimum and maximum value in each column,
* mean() and median(): get the mean and median value in each column,
* std() and var(): get the standard deviation and variance in each column.

## Get the Non-NA values in each column

In [15]:
penguins.count()

species              344
island               344
bill_length_mm       342
bill_depth_mm        342
flipper_length_mm    342
body_mass_g          342
sex                  333
year                 344
dtype: int64

## Get the minimum value in each column with numerical values

In [16]:
penguins.select_dtypes('number').min() #ask about this

bill_length_mm         32.1
bill_depth_mm          13.1
flipper_length_mm     172.0
body_mass_g          2700.0
year                 2007.0
dtype: float64

## Grouping
Our penguins data is naturally split into different groups: there are three different species, two sexes (with some missing values), and three islands. Often, we want to calculate a certain statistic for each group. For example, suppose we want to calculate the average flipper length per species. How would we do this “by hand”?

---
1. We start with our data and notice there are multiple species in the species column.

2. We split our original table to group all observations from the same species together.

3. We calculate the average flipper length for each of the groups we formed.

4. Then we combine the values for average flipper length per species into a single table.
---
This is known as the **Split-Apply-Combine strategy**. This strategy follows the three steps we explained above:

1. **Split:** Split the data into logical groups (e.g. species, sex, island, etc.)

2. **Apply:** Calculate some summary statistic on each group (e.g. average flipper length by species, number of individuals per island, body mass by sex, etc.)

3. **Combine:** Combine the statistic calculated on each group back together.

For a `pandas.DataFrame` or `pandas.Series`, we can use the *groupby()* method to split (i.e. group) the data into different categories.

The general syntax for `groupby()`:

df.groupby(columns_to_group_by).summary_method()

## Example
If we don’t use `groupby()` and directly apply the `mean()` method to our flipper length column, we obtain the average of all the values in the column:

In [23]:
penguins['flipper_length_mm'].mean()

200.91520467836258

In [24]:
#groupby() methoid without specifying what we wish to calculate on each group
penguins.groupby('species')['flipper_length_mm']

We get a `GroupBy` object, which is an intermediate step. It doesn’t perform the actual calculations until we specify an operation:

In [25]:
#average flipper length per species
penguins.groupby('species')['flipper_length_mm'].mean()

species
Adelie       189.953642
Chinstrap    195.823529
Gentoo       217.186992
Name: flipper_length_mm, dtype: float64

Let’s recap what happened in that line (the . can be read as “and then…”):

* start with the penguins data frame, and then…
* use `groupby()` to group the data frame by species values, and then…
* select the 'flipper_length_mm' column, and then…
* calculate the mean() of this column with respect to the groups.

Notice that the name of the series is the same as the column on which we calculated the summary statistic. We can easily update this using the rename() method:

In [26]:
# Average flipper length per species
avg_flipper = (penguins.groupby('species')
                        ['flipper_length_mm']
                        .mean()
                        .rename('mean_flipper_length')
                        .sort_values(ascending=False)
                        )
avg_flipper 

species
Gentoo       217.186992
Chinstrap    195.823529
Adelie       189.953642
Name: mean_flipper_length, dtype: float64

# Grouping and NA values
--- 
Notice first that 11 penguins have no recorded sex. We can check this using:

* size: returns the number of observations in a pandas.Series
* count(): returns the number of non-NA observations in the series

In [27]:
# Number of NA observations
penguins['sex'].size - penguins['sex'].count()

11

By default, `groupby()` leaves out the rows where the grouping column has NA values. When we group by sex and count the number of observations (rows) we are left with, the 11 NA values are not included:

In [28]:
penguins.groupby('sex').size()

sex
female    165
male      168
dtype: int64

To keep the NA values as their own group, we use the `dropna=False` parameter:

In [29]:
penguins.groupby('sex', dropna=False).size()

sex
female    165
male      168
NaN        11
dtype: int64

## Check-in
1. run > penguins.groupby(['island','year']).count() & penguins.groupby(['island','year']).size()
2. Discuss the differences between each output. What method would give you the number of penguins surveyed on each island and year combination?

In [30]:
#run the code
penguins.groupby(['island','year']).count()

species  bill_length_mm  bill_depth_mm  flipper_length_mm  \
island    year                                                              
Biscoe    2007       44              44             44                 44   
          2008       64              64             64                 64   
          2009       60              59             59                 59   
Dream     2007       46              46             46                 46   
          2008       34              34             34                 34   
          2009       44              44             44                 44   
Torgersen 2007       20              19             19                 19   
          2008       16              16             16                 16   
          2009       16              16             16                 16   

                body_mass_g  sex  
island    year                    
Biscoe    2007           44   43  
          2008           64   63  
          2009           59   57  
Dream     2007           46   45  
          2008           34   34  
          2009           44   44  
Torgersen 2007           19   15  
          2008           16   16  
          2009           16   16

In the groupby method we are gettinf the counts 

In [31]:
#run the code
penguins.groupby(['island','year']).size()

island     year
Biscoe     2007    44
           2008    64
           2009    60
Dream      2007    46
           2008    34
           2009    44
Torgersen  2007    20
           2008    16
           2009    16
dtype: int64